# Chapter 18 — Changing What Pi Knows

**Companion to *Pi Agents*** · [`Pi Agents` chapter 18](https://programmer.ie/books/pi/18-chapter/)

| | |
|---|---|
| Chapter | `18-chapter.md` · weight 18 · `/books/pi/18-chapter/` |
| Notebook | `notebooks/pi/18-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does a section derived from *active* tools disappear when the tool is disabled,
while `getAllTools()` still reports it?

The chapter's point: the system prompt describes the **active** set, not the
registered set. `getAllTools()` reports everything registered; `getActiveTools()`
reports what the model can actually call. The two disagree exactly when a tool
is registered but not active.


## What this notebook establishes

- The `tool_guidance` section is derived from **active tools only** — a tool that is not active is not described.
- A section derived from an active tool **disappears when the tool is disabled** and **reappears when re-enabled** — the condition is evaluated per run.
- `getAllTools()` reports the **registered** set; `getActiveTools()` reports the **active** set. They disagree exactly when a tool is registered but not active.
- `agent_before_settle` can append a `custom_message` entry and return `continue: true` — exactly one continuation, not a loop.

## What this notebook does **not** establish

- **`session.systemPrompt` omits the section on 1.0.4** — observed, and the reason the provider's copy is read instead of the session's (`ch18-lim1`).
- **No token cost measured** for the tool guidance section (`ch18-lim2`).
- **The settle-guard only nudges once** — the `custom_message` entry is a guard against an infinite loop.
- **The chapter's tests use `--tools` to filter at registration**, but the active/disabled case uses `setActiveTools()` which is different — `tools` on the session filters at registration, so `run_checks` would never be registered there.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real sessions walking active → disabled → re-enabled, reading the provider's system prompt via `systemPromptSeenByProvider`.
* **Evidence scope:** `in_process_runtime`. The active-set rule and `agent_before_settle` behaviour are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(18))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/18-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the tool_guidance section comes from active tools

`prompt-customizer` reads `options.selectedTools` and builds a section only for
tools that are active.

In [3]:
KNOW = pinb.driver_source("ch18-knowledge.ts")
know = pinb.driver(KNOW, label="ch18-knowledge", timeout=300)
print(pinb.md_table(
    ["session", "has tool_guidance?", "has read?", "has bash?"],
    [["all tools active", "yes" if know["promptCustomizer"]["hasToolGuidance"] else "no", "yes" if know["promptCustomizer"]["hasRead"] else "no", "yes" if know["promptCustomizer"]["hasBash"] else "no"],
     ["only read active", "yes" if know["promptCustomizerOnlyRead"]["hasRead"] else "no", "yes" if know["promptCustomizerOnlyRead"]["hasRead"] else "no", "yes" if know["promptCustomizerOnlyRead"]["hasBash"] else "no"]],
))

| session | has tool_guidance? | has read? | has bash? |
|---|---|---|---|
| all tools active | yes | yes | yes |
| only read active | yes | yes | no |


In [4]:
pinb.show_checks([
    pinb.check("tool_guidance section is delivered when tools are active",
               know["promptCustomizer"]["hasToolGuidance"], "section present"),
    pinb.check("only active tools are described: read yes, bash no",
               know["promptCustomizerOnlyRead"]["hasRead"] and not know["promptCustomizerOnlyRead"]["hasBash"], "read yes, bash no"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------  -----------------
PASS  tool_guidance section is delivered when tools are active  section present
PASS  only active tools are described: read yes, bash no  read yes, bash no

2/2 assertions held.


## The review-guard section: appears when `run_checks` is active, disappears with it

`review-guard-section` checks `pi.getActiveTools()` for `run_checks`. It deletes
the section rather than leaving stale content.

In [5]:
print(pinb.md_table(
    ["session", "has review_guard section?"],
    [["tools + review_guard_section (run_checks active)", "yes" if know["reviewGuardWithTool"]["hasSection"] else "no"],
     ["review_guard_section alone (run_checks not registered)", "yes" if know["reviewGuardWithoutTool"]["hasSection"] else "no"],
     ["tools + review_guard_section, tools=[read, run_checks] (active)", "yes" if know["reviewGuardActive"]["hasSection"] else "no"],
     ["tools + review_guard_section, tools=[read] (disabled)", "yes" if know["reviewGuardDisabled"]["hasSection"] else "no"],
     ["tools + review_guard_section, tools=[read, run_checks] (re-enabled)", "yes" if know["reviewGuardReenabled"]["hasSection"] else "no"]],
))

| session | has review_guard section? |
|---|---|
| tools + review_guard_section (run_checks active) | yes |
| review_guard_section alone (run_checks not registered) | no |
| tools + review_guard_section, tools=[read, run_checks] (active) | yes |
| tools + review_guard_section, tools=[read] (disabled) | no |
| tools + review_guard_section, tools=[read, run_checks] (re-enabled) | yes |


In [6]:
pinb.show_checks([
    pinb.check("with tools, run_checks is active and section appears",
               know["reviewGuardWithTool"]["hasSection"], "appears"),
    pinb.check("without tools, section does not appear",
               not know["reviewGuardWithoutTool"]["hasSection"], "absent"),
    pinb.check("with tools but only read active, section is absent",
               not know["reviewGuardDisabled"]["hasSection"], "absent when disabled"),
    pinb.check("re-enabling run_checks makes section reappear",
               know["reviewGuardReenabled"]["hasSection"], "reappears"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------  --------------------
PASS  with tools, run_checks is active and section appears  appears
PASS  without tools, section does not appear  absent
PASS  with tools but only read active, section is absent  absent when disabled
PASS  re-enabling run_checks makes section reappear  reappears

4/4 assertions held.


## The accessor probe: `getAllTools()` vs `getActiveTools()`

The two accessors disagree exactly when a tool is registered but not active.

In [7]:
print(pinb.md_table(
    ["accessor", "includes run_checks?"],
    [["before narrowing (active)", "yes" if "run_checks" in (know["accessor"]["beforeNarrow"] or []) else "no"],
     ["getAllTools() (registered)", "yes" if "run_checks" in (know["accessor"]["registered"] or []) else "no"],
     ["getActiveTools() after narrowing (active)", "yes" if "run_checks" in (know["accessor"]["active"] or []) else "no"]],
))

| accessor | includes run_checks? |
|---|---|
| before narrowing (active) | yes |
| getAllTools() (registered) | yes |
| getActiveTools() after narrowing (active) | no |


In [8]:
pinb.show_checks([
    pinb.check("before narrowing, run_checks is active",
               "run_checks" in (know["accessor"]["beforeNarrow"] or []), "active initially"),
    pinb.check("getAllTools() reports run_checks (registered)",
               "run_checks" in (know["accessor"]["registered"] or []), "registered"),
    pinb.check("getActiveTools() does NOT report run_checks (not active)",
               "run_checks" not in (know["accessor"]["active"] or []), "not active after narrowing"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------  --------------------------
PASS  before narrowing, run_checks is active  active initially
PASS  getAllTools() reports run_checks (registered)  registered
PASS  getActiveTools() does NOT report run_checks (not active)  not active after narrowing

3/3 assertions held.


## Settle guard: `agent_before_settle` buys exactly one continuation

If the model answers without running `run_checks`, the guard nudges once with a
`custom_message` entry and `continue: true`. A second answer without checks is
not nudged again.

In [9]:
print(pinb.md_table(
    ["scenario", "requests", "nudge sent?"],
    [["model never runs run_checks", know["settleNoChecks"]["callCount"], "yes" if know["settleNoChecks"]["hasNudge"] else "no"],
     ["model runs run_checks", know["settleWithChecks"]["callCount"], "yes" if know["settleWithChecks"]["hasNudge"] else "no"]],
))

| scenario | requests | nudge sent? |
|---|---|---|
| model never runs run_checks | 2 | yes |
| model runs run_checks | 2 | no |


In [10]:
pinb.show_checks([
    pinb.check("never running checks -> exactly 2 requests (one continuation)",
               know["settleNoChecks"]["callCount"] == 2 and know["settleNoChecks"]["hasNudge"], "2 requests, nudge sent"),
    pinb.check("running checks -> 2 requests, no nudge",
               know["settleWithChecks"]["callCount"] == 2 and not know["settleWithChecks"]["hasNudge"], "2 requests, no nudge"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------------  ----------------------
PASS  never running checks -> exactly 2 requests (one continuation)  2 requests, nudge sent
PASS  running checks -> 2 requests, no nudge  2 requests, no nudge

2/2 assertions held.


## The chapter's own tests

All seven tests from the chapter's test file.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [11]:
PATTERNS = 'ch18-knowledge/knowledge.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch18-knowledge/knowledge.test.ts', show="active")

$ node --test ch18-knowledge/knowledge.test.ts
  PASS  7 passed, 0 failed, 0 skipped  in 1025 ms

  tests matching 'active':
    ok   prompt-customizer: the tool guidance section is derived from the tools that are active
    ok   prompt-customizer: a tool that is not active is not described
    ok   review-guard section appears when run_checks is active and disappears with it
    ok   review-guard section tracks the active tool set, not the registered one
    ok   getAllTools would have reported the disabled tool; getActiveTools does not


## Your turn: predict, then run

**Predict first.** What happens if `agent_before_settle` returns `continue: true`
**unconditionally**? Does it loop forever?

**Then change one input.** The `review-guard-section` uses `delete
event.systemPromptOptions.sections.review_guard` when the tool is not active.
What happens if it leaves the section but sets it to an empty string instead?

**Predict the boundary.** The chapter says `session.systemPrompt` omits the
section on 1.0.4. The driver reads `systemPromptSeenByProvider` — what does
`session.systemPrompt` actually contain at this pin?

In [12]:
print("Predictions:")
print("  1. Unconditional continue: true -> infinite loop (the chapter guards against this).")
print("  2. Empty string section: probably still sent to provider as empty section.")
print("  3. session.systemPrompt omits the section; provider copy is the ground truth.")
print()
print("Observed above:")
print(f"  active tools: read={know["promptCustomizerOnlyRead"]["hasRead"]}, bash={know["promptCustomizerOnlyRead"]["hasBash"]}")
print(f"  disabled run_checks: section={"yes" if know["reviewGuardDisabled"]["hasSection"] else "no"}")
print(f"  accessor: registered={len(know["accessor"]["registered"] or [])}, active={len(know["accessor"]["active"] or [])}")
print(f"  settle no checks: {know["settleNoChecks"]["callCount"]} requests, nudge={know["settleNoChecks"]["hasNudge"]}")
print()
assert know["promptCustomizerOnlyRead"]["hasRead"] and not know["promptCustomizerOnlyRead"]["hasBash"]
assert not know["reviewGuardDisabled"]["hasSection"]
assert know["reviewGuardReenabled"]["hasSection"]
assert "run_checks" in (know["accessor"]["registered"] or [])
assert "run_checks" not in (know["accessor"]["active"] or [])
assert know["settleNoChecks"]["callCount"] == 2
print("held: active set controls the prompt, registered set is larger, settle guard nudges once")

Predictions:
  1. Unconditional continue: true -> infinite loop (the chapter guards against this).
  2. Empty string section: probably still sent to provider as empty section.
  3. session.systemPrompt omits the section; provider copy is the ground truth.

Observed above:
  active tools: read=True, bash=False
  disabled run_checks: section=no
  accessor: registered=11, active=1
  settle no checks: 2 requests, nudge=True

held: active set controls the prompt, registered set is larger, settle guard nudges once


## Interpretation

What the model knows comes from three sources, and each has a boundary:

| Source | What it contributes | The boundary |
|---|---|---|
| `tool_guidance` section | One line per active built-in tool | Inactive tools are not described — the model cannot call them |
| Extension sections (`before_agent_start`) | Arbitrary text from extensions | Evaluated per run; stale sections must be deleted |
| `agent_before_settle` | `custom_message` entries + `continue: true` | **One continuation max** — the entry is the loop guard |

The practical rules:
* **Describe what the model can call** — the active set, not the registered set. `getAllTools()` is for tooling; `getActiveTools()` is for the prompt.
* **Delete stale sections** — a section written for a previous turn outlives the condition that produced it. Use `delete`, not empty string.
* **`agent_before_settle` is the last actionable boundary** — it can append entries and buy one more request. `agent_settled` is notification-only.
* **The provider's copy is the ground truth** — `session.systemPrompt` may omit sections; always read what the provider actually received.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real sessions walking active → disabled → re-enabled, reading the provider's system prompt via `systemPromptSeenByProvider`.

**Evidence scope:** `in_process_runtime`. The active-set rule and `agent_before_settle` behaviour are Pi's.

### What was read or run

- **Ran** `drivers/ch18-knowledge.ts`: ten real sessions covering prompt-customizer, review-guard-section, accessor probe, and settle-guard.
- **Ran** `ch18-knowledge/knowledge.test.ts` (7 tests).
- **Read** `examples/evidence.json`, chapter 18 rows (7 claims).

### Limitations

- **`session.systemPrompt` omits the section on 1.0.4** — observed, and the reason the provider's copy is read instead (`ch18-lim1`).
- **No token cost measured** for the tool guidance section (`ch18-lim2`).
- **The settle-guard only nudges once** — the `custom_message` entry is the loop guard.
- **`tools` on the session filters at registration** — the active/disabled case uses `setActiveTools()` which produces a different boundary (registered but not active).

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
